In [1]:

import numpy as np

from scripts.data_filter import filter_dataframe
from scripts.data_loader import load_dataframe
from scripts.utils import RF_PARAM_5G, NETWORK_TYPE

filename = "5G_data_2023.mat"

# Series of random seeds for reproducability
random_seeds = np.loadtxt("data/random_seeds.csv", dtype=int)

# load the dataframe from saved file or 'raw' matlab file
df = load_dataframe(filename, NETWORK_TYPE._5G)

# # Drop unused columns to save space
matrix_cols_to_drop = ["toa_pps", "toa_cir", "toa_cov", "campaign_id"]
df["measurements_matrix"] = df["measurements_matrix"].apply(
    lambda x: x.drop(columns=matrix_cols_to_drop)
)

operator_choice = [10]
selected_campaigns = list(range(1, 11))
rf_param = RF_PARAM_5G.RSRQ

# Data filtering
df_orig = filter_dataframe(
    df=df,
    operators=operator_choice,
    include_columns=[
        "pci",
        "beam_index",
        "nr_arfcn",
        "operator_id",
        "sinr",
        "rsrq"
    ],
    campaigns=selected_campaigns,
)


Loaded dataframe from .h5 file: /Users/andreres/Documents/UIO/master/thesis/dev/5G_localization/data/dataframe_cache/5G_data_2023.h5


/Users/andreres/Documents/UIO/master/thesis/dev/5G_localization/scripts/data_filter.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["measurements_matrix"] = df["measurements_matrix"].apply(


In [2]:
df = df_orig.sample(500)

In [3]:
from scripts.beamforming import get_best_beam

df["best_beam"] = df["measurements_matrix"].apply(
    lambda x: get_best_beam(x, rf_param)
)

In [4]:
df

,lat,lng,measurements_matrix,campaign_id,best_beam
3025,41.898439,12.428082,pci beam_index nr_arfcn operator_id ...,7,"(-109.0, 1.0, 643296.0, 10.0)"
3829,41.899479,12.427502,pci beam_index nr_arfcn operator_id ...,9,"(-109.0, 4.0, 643296.0, 10.0)"
4308,41.899323,12.427651,pci beam_index nr_arfcn operator_id ...,10,"(-109.0, 4.0, 643296.0, 10.0)"
4081,41.896589,12.428813,pci beam_index nr_arfcn operator_id ...,9,"(76.0, 4.0, 643296.0, 10.0)"
3699,41.898357,12.430067,pci beam_index nr_arfcn operator_id ...,9,"(-65.0, 6.0, 643296.0, 10.0)"
...,...,...,...,...,...
4556,41.898297,12.428584,pci beam_index nr_arfcn operator_id ...,10,"(-108.0, 5.0, 643296.0, 10.0)"
3178,41.898315,12.430048,pci beam_index nr_arfcn operator_id ...,8,"(-108.0, 4.0, 643296.0, 10.0)"
2848,41.897562,12.427931,pci beam_index nr_arfcn operator_id ...,7,"(-108.0, 3.0, 643296.0, 10.0)"
3739,41.898548,12.429669,pci beam_index nr_arfcn operator_id ...,9,"(-109.0, 1.0, 643296.0, 10.0)"


In [54]:
from scripts.utils import dataset_tp_rp_split, extract_unique_npcis
from scripts.weighted_coverage import create_point_matrix, compute_weights
from scripts.beamforming import filter_best_beam
import pandas as pd

df_tp, df_rp = dataset_tp_rp_split(df, 0.3, 420)
df_rp_ctrl = df_rp.copy()
df_rp_filter = df_rp.copy()

all_pcis = extract_unique_npcis(df['measurements_matrix'])

tp = df_tp.iloc[7]

best_beam = tp["best_beam"]
best_beam

tp = pd.DataFrame([tp])

# Calculate the point matricies with rf values for all RPs and TP using only the best beam PCI
m_rp, idx_rp = create_point_matrix(df_rp, [best_beam], rf_param)
m_tp, idx_tp = create_point_matrix(tp, [best_beam], rf_param)
m_tp_ctrl, idx_tp_ctrl = create_point_matrix(tp, all_pcis, rf_param)
W, idx_sort = compute_weights(m_rp, idx_rp, m_tp, idx_tp)

# Calculate the point matricies with rf values for all RPs and TP using all PCIs

m_rp_ctrl, idx_rp_ctrl = create_point_matrix(df_rp_ctrl, all_pcis, rf_param)
m_tp_ctrl, idx_tp_ctrl = create_point_matrix(tp, all_pcis, rf_param)
W_ctrl, idx_sort_ctrl = compute_weights(m_rp_ctrl, idx_rp_ctrl, m_tp_ctrl, idx_tp_ctrl)

# Calculate the point matricies with rf values for all RPs and TP using all PCIs, with filtered measurements matrix
tp_filter = tp.copy()

# fitler the dataframes measurements matrix
tp_filter.loc[:, "measurements_matrix"] = tp_filter.loc[:, "measurements_matrix"].apply(
    lambda x: filter_best_beam(x, rf_param)
)
df_rp_filter.loc[:, "measurements_matrix"] = df_rp_filter.loc[:, "measurements_matrix"].apply(
    lambda x: filter_best_beam(x, rf_param)
)
m_rp_filter, idx_rp_filter = create_point_matrix(df_rp_filter, all_pcis, rf_param)
m_tp_filter, idx_tp_filter = create_point_matrix(tp_filter, all_pcis, rf_param)
W_filter, idx_sort_filter = compute_weights(m_rp_filter, idx_rp_filter, m_tp_filter, idx_tp_filter)

print(f"""
Control (without using only best beam)
Selected RPs indecies \t {idx_sort_ctrl[0, :10]}

Test (using only best beam)
Selected RPs indecies \t {idx_sort[0, :10]}

Test (using best beam filtering)
Selected RPs indecies \t {idx_sort_filter[0, :10]}
""")


Control (without using only best beam)
Selected RPs indecies 	 [262 113 335  46 231 172  94  22 128 143]

Test (using only best beam)
Selected RPs indecies 	 [228 333 239 231  16  37  34 170 295  46]

Test (using best beam filtering)
Selected RPs indecies 	 [228 333 231  16  37  34 170 295  46  35]



In [70]:
from scipy.spatial.distance import cdist
from sklearn.decomposition import PCA
from scripts.weighted_coverage import wknn_one


def compute_weights_pca(m_rfp_pca, m_tp_pca):
    """
    Compute weights using PCA-transformed data
    """
    # Compute Euclidean distances in the PCA space
    D = cdist(m_tp_pca, m_rfp_pca, metric="euclidean")

    # Sort distances and compute weights
    idx_sort = np.argsort(D, axis=1)
    D_sort = np.take_along_axis(D, idx_sort, axis=1)

    # Avoid division by zero
    min_nonzero_distance = np.min(D[D > 0]) if np.any(D > 0) else 0.1
    D_sort[D_sort == 0] = min_nonzero_distance / 20

    W = 1.0 / D_sort
    return W, idx_sort


def pca_strategy(df_tp: pd.DataFrame, df_rp: pd.DataFrame, pcis: list[tuple], rf_param: RF_PARAM_5G):
    m_rp_full, idx_rp_full = create_point_matrix(df_rp, pcis, rf_param)
    m_tp_full, idx_tp_full = create_point_matrix(tp, pcis, rf_param)

    pca = PCA(n_components=0.95)
    pca.fit(m_rp_full)
    m_rp_pca = pca.transform(m_rp_full)
    m_tp_pca = pca.transform(m_tp_full)

    W_pca, idx_sort_pca = compute_weights_pca(m_rp_pca, m_tp_pca)

    _, errors = wknn_one(df_tp, df_rp, W_pca, idx_sort_pca, k=2)

    return errors


def normal_strategy(df_tp: pd.DataFrame, df_rp: pd.DataFrame, pcis: list[tuple], rf_param: RF_PARAM_5G):
    m_rp_full, idx_rp_full = create_point_matrix(df_rp, pcis, rf_param)
    m_tp_full, idx_tp_full = create_point_matrix(tp, pcis, rf_param)

    W, idx_sort = compute_weights(m_rp_full, idx_rp_full, m_tp_full, idx_tp_full)

    _, errors = wknn_one(df_tp, df_rp, W, idx_sort_pca, k=2)

    return errors


res_pca = pca_strategy(tp, df_rp, all_pcis, rf_param)
res_normal = normal_strategy(tp, df_rp, all_pcis, rf_param)

print(f"""
PCA {res_pca}
Normal {res_normal}
""")


PCA [242.66366822]
Normal [242.66366822]



In [61]:

print(f"""
Control (without using only best beam)
Selected RPs indecies \t {idx_sort_ctrl[0, :10]}

Test (using only best beam)
Selected RPs indecies \t {idx_sort[0, :10]}

Test (using best beam filtering)
Selected RPs indecies \t {idx_sort_filter[0, :10]}

Test (PCA)
Selected RPs indecies \t {idx_sort_pca[0, :10]}
""")


Control (without using only best beam)
Selected RPs indecies 	 [262 113 335  46 231 172  94  22 128 143]

Test (using only best beam)
Selected RPs indecies 	 [228 333 239 231  16  37  34 170 295  46]

Test (using best beam filtering)
Selected RPs indecies 	 [228 333 231  16  37  34 170 295  46  35]

Test (PCA)
Selected RPs indecies 	 [262 113  46 335 231 128 143  22 314  55]



In [29]:
idx_sort_ctrl

array([[262, 113, 335,  46, 231, 172,  94,  22, 128, 143, 314,  55, 189,
         61, 112,  88, 278,  85,  14,   6, 211, 320,  66,  93,  97,  58,
        100, 206, 273,  32, 296, 274, 210, 234,   3, 155, 158, 135,  35,
         91, 261, 333,  56,  63,  65,  34, 215, 197, 268, 207, 131, 257,
         45, 228, 311, 299,  18, 279, 251, 308, 179, 176,  51, 270, 146,
        111, 341, 166, 325,  90, 142, 328, 235,  75,   1, 133,  16, 117,
          9,   8,  50,  37,  25,  23,  19,  84,  69, 263, 321, 301, 174,
        219,  48,  41, 239, 269, 173,  89, 185, 253, 339, 288, 304, 336,
        309, 227, 184, 334, 170, 192,   4, 106, 313, 200, 130, 233, 258,
         28,  78, 298, 290, 161, 327,  77, 238, 144, 107, 244, 340, 295,
        319, 229, 127,  67, 165, 245, 259, 240, 191, 241,  54, 226, 285,
        324, 187, 208, 149,  52, 267, 110,  10, 150, 265, 186, 284, 249,
        164,  24, 137, 202,  30,  11, 300,  43, 109,  73,  13, 305, 283,
        255, 108, 156, 193, 152, 322,  57, 154, 330

In [30]:
W_ctl

array([[0.78283145, 0.71319321, 0.67973016, 0.63960827, 0.62756969,
        0.58387194, 0.57625116, 0.54815524, 0.54813402, 0.54718274,
        0.53107916, 0.52860722, 0.51358077, 0.48938416, 0.48031329,
        0.48023471, 0.47495583, 0.47190997, 0.46752064, 0.46418147,
        0.4612759 , 0.45880577, 0.45268701, 0.44261762, 0.43528258,
        0.43496194, 0.43306809, 0.42600693, 0.42032788, 0.41845487,
        0.41837591, 0.41737   , 0.41248453, 0.40867709, 0.40867195,
        0.40775113, 0.40440745, 0.4002615 , 0.39605852, 0.39283177,
        0.38870491, 0.38668905, 0.38505508, 0.38319877, 0.38305085,
        0.38205646, 0.37894143, 0.3708124 , 0.36604568, 0.36575196,
        0.3636421 , 0.3580243 , 0.35728345, 0.35422315, 0.35359995,
        0.3516544 , 0.35140006, 0.34910115, 0.34886201, 0.34601302,
        0.34489751, 0.34426915, 0.34276391, 0.34183169, 0.34023354,
        0.3394283 , 0.33882438, 0.33867316, 0.33755546, 0.3346026 ,
        0.3329356 , 0.33212015, 0.32408015, 0.32

In [55]:
compare_ctrl = np.hstack((idx_sort_ctrl.T, W_ctl.T))
compare_ctrl = compare_ctrl[compare_ctrl[:, 0].argsort()]

compare_filter = np.hstack((idx_sort_filter.T, W_ctl.T))
compare_filter = compare_filter[compare_filter[:, 0].argsort()]

compare = np.hstack((idx_sort.T, W.T))
compare = compare[compare[:, 0].argsort()]

compare

total = np.hstack((compare, compare_ctrl, compare_filter))

total

array([[0.00000000e+000, 5.07614213e-002, 0.00000000e+000,
        2.55157013e-001, 0.00000000e+000, 2.65903026e-001],
       [1.00000000e+000, 4.25331201e-002, 1.00000000e+000,
        3.22364053e-001, 1.00000000e+000, 2.29137005e-001],
       [2.00000000e+000, 5.56268465e-309, 2.00000000e+000,
        1.56021722e-001, 2.00000000e+000, 2.29666735e-001],
       ...,
       [3.39000000e+002, 6.30517024e-002, 3.39000000e+002,
        2.96411345e-001, 3.39000000e+002, 2.88679629e-001],
       [3.40000000e+002, 1.79211470e-001, 3.40000000e+002,
        2.82651485e-001, 3.40000000e+002, 1.93969788e-001],
       [3.41000000e+002, 3.43160308e-002, 3.41000000e+002,
        3.38824383e-001, 3.41000000e+002, 8.42711915e-003]])

In [42]:
compare_ctrl

array([[0.00000000e+00, 2.55157013e-01],
       [1.00000000e+00, 3.22364053e-01],
       [2.00000000e+00, 1.56021722e-01],
       [3.00000000e+00, 4.08671952e-01],
       [4.00000000e+00, 2.89968171e-01],
       [5.00000000e+00, 2.01248340e-01],
       [6.00000000e+00, 4.64181466e-01],
       [7.00000000e+00, 1.56969199e-01],
       [8.00000000e+00, 3.17079470e-01],
       [9.00000000e+00, 3.17529960e-01],
       [1.00000000e+01, 2.74626273e-01],
       [1.10000000e+01, 2.70418791e-01],
       [1.20000000e+01, 8.42711915e-03],
       [1.30000000e+01, 2.69394040e-01],
       [1.40000000e+01, 4.67520640e-01],
       [1.50000000e+01, 2.04644840e-01],
       [1.60000000e+01, 3.20767606e-01],
       [1.70000000e+01, 2.24403985e-01],
       [1.80000000e+01, 3.51400056e-01],
       [1.90000000e+01, 3.06531988e-01],
       [2.00000000e+01, 2.13664248e-01],
       [2.10000000e+01, 2.44751872e-01],
       [2.20000000e+01, 5.48155244e-01],
       [2.30000000e+01, 3.08805995e-01],
       [2.400000